# Per-Stage v3 Adapters vs GPT-5.5 (low reasoning) — proper n=120 benchmark

The real head-to-head for the two dedicated v3 adapters, at a sample size that's
actually trustworthy (n=120, not the n=25 that gave a noisy 84-vs-80 result earlier).
One Run All, loads Qwen3-VL once, attaches BOTH adapters as named peft adapters and
switches between them for scoring — no redundant 8B model loads.

**Comparisons, each model on its own task:**

| Task | Models compared | Ground truth |
|---|---|---|
| Stage 13 (entity validation, keep/remove) | GPT-5.5-low, Qwen base, Qwen+v3-stage13 | Gupta frozen TEST sheets (real, genuinely held out) |
| Stage 12 (relation validation, connected y/n) | GPT-5.5-low, Qwen base, Qwen+v3-relation | PID2Graph (real), seed-disjoint from training |

**Honesty notes, read before trusting the numbers:**
- Stage 13's eval is on Gupta's real frozen TEST sheets — no adapter has ever trained
  on these. This is a genuinely held-out check.
- Relation's eval uses a different random seed than training (8181 vs training's 777)
  sampled from the same source trees (OPEN100 + Dataset PID) — meaningfully disjoint
  in practice, but not a hard train/test split the way Gupta's test sheets are. Read
  as "trustworthy signal," not "airtight held-out proof."
- GPT-5.5 gets `max_completion_tokens=2000` (a prior run at 32-64 tokens silently
  starved on its own reasoning and returned empty replies, scored as "undecided" —
  fixed here from the start).
- `RELATION_ADAPTER_PATH` defaults to `v3-relation/latest`, i.e. whatever checkpoint
  exists when you run this — the relation phase may still be training/paused. Update
  it to a specific `epoch_N` once that phase fully completes for the final number.

## 1. Config

In [5]:
HF_TOKEN = "paste-your-hf-token-here"
OPENAI_API_KEY = "paste-your-openai-key-here"

GPT_REASONING_EFFORT = "low"
DATA_REPO = "timthy45/pnid-extraction-datasets"
CKPT_REPO = "timthy45/qwen3vl-pnid-domain-base"
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

STAGE13_ADAPTER_PATH = "v3-stage13/latest"   # completed, 3 epochs
RELATION_ADAPTER_PATH = "v3-relation/latest"  # whatever checkpoint exists now

N_PER_TASK = 120
RESULTS_PATH_IN_REPO = "benchmarks/per_stage_v3_vs_gpt55.csv"

assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "paste-your-hf-token-here"
assert OPENAI_API_KEY.startswith("sk-") and OPENAI_API_KEY != "paste-your-openai-key-here"

## 2. Install + GPU check

In [6]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q pytesseract huggingface_hub openai peft
!pip uninstall -y torchao -q

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch
assert torch.cuda.is_available(), "No GPU - set Runtime type first"
print(torch.cuda.get_device_name(0))

NVIDIA A100-SXM4-80GB


## 3. Data from HF (resumable - retries transient CDN signing failures)

Same retry pattern that fixed today's real "invalid key pair id" xet-bridge errors:
force a fresh signed URL and back off, up to 5 attempts per file.

In [7]:
import zipfile, time, json, random
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA = Path("/content/data")
DATA.mkdir(exist_ok=True)

def fetch_with_retry(filename, max_attempts=5):
    for attempt in range(max_attempts):
        try:
            return hf_hub_download(repo_id=DATA_REPO, filename=filename,
                                   repo_type="dataset", token=HF_TOKEN,
                                   force_download=(attempt > 0))
        except Exception as e:
            print(f"  [retry {attempt+1}/{max_attempts}] "
                 f"{type(e).__name__}: {str(e)[:120]}")
            time.sleep(15)
    raise RuntimeError(f"download of {filename} failed after {max_attempts} retries")

for fname, extract_to in [
    ("gupta_pid/PID_Dataset.zip", DATA / "gupta"),
    ("pid2graph/PID2Graph.zip", DATA / "pid2graph"),
]:
    if extract_to.exists() and any(extract_to.iterdir()):
        print(f"{extract_to} already extracted"); continue
    zp = fetch_with_retry(fname)
    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(extract_to)
    print(f"extracted {fname}")

GUPTA_RAW = DATA / "gupta" / "PID_Dataset" / "0__raw_data"
P2G_PATCHED = DATA / "pid2graph" / "PID2Graph" / "Patched"
OPEN100_DIR = P2G_PATCHED / "PID2Graph OPEN100"
DATASETPID_DIR = P2G_PATCHED / "Dataset PID"
for p in [GUPTA_RAW / "sheets" / "test", OPEN100_DIR]:
    assert p.exists(), f"missing: {p}"
print("data ready")

/content/data/gupta already extracted
/content/data/pid2graph already extracted
data ready


## 4. Build both eval pools (n=120 each, built once, shared by all models)

In [8]:
import xml.etree.ElementTree as ET
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

def load_boxes(label_path, W, H):
    boxes = []
    for line in Path(label_path).read_text().splitlines():
        if line.strip():
            parts = line.split()
            cx, cy, w, h = (float(v) for v in parts[1:5])
            boxes.append([(cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H])
    return boxes

def overlaps_any(box, boxes):
    return any(box[0] < b[2] and box[2] > b[0] and box[1] < b[3] and box[3] > b[1] for b in boxes)

rng13 = random.Random(2468)   # fresh seed, distinct from training (13) and the live eval (9999)

def build_stage13_pool(n_target):
    examples = []
    test_sheets = sorted((GUPTA_RAW / "sheets" / "test").glob("*.*"))
    rng13.shuffle(test_sheets)
    for sheet_path in test_sheets:
        if len(examples) >= n_target:
            break
        img = Image.open(sheet_path).convert("RGB")
        W, H = img.size
        label_path = GUPTA_RAW / "labels" / "test" / f"{sheet_path.stem}.txt"
        boxes = load_boxes(label_path, W, H)
        if not boxes:
            continue
        rng13.shuffle(boxes)
        for real_box in boxes[:6]:
            if len(examples) >= n_target:
                break
            m = 60
            cbox = [max(0, real_box[0]-m), max(0, real_box[1]-m),
                   min(W, real_box[2]+m), min(H, real_box[3]+m)]
            cb = [int(real_box[0]-cbox[0]), int(real_box[1]-cbox[1]),
                 int(real_box[2]-cbox[0]), int(real_box[3]-cbox[1])]
            crop = img.crop([int(v) for v in cbox])
            p = (f"A symbol detector claims there is a P&ID equipment "
                f"symbol at [{cb}] in this crop (pixel coords, "
                f"top-left origin). Verify against the pixels: is "
                f"there really a discrete equipment symbol there "
                f"(valve, instrument, fitting)? Answer keep or remove.")
            examples.append({"crop": crop, "prompt": p, "is_real": True})

            absence_type = rng13.choice(["empty", "shifted", "wrong_size"])
            bad_box = None
            if absence_type == "empty":
                for _ in range(80):
                    size = rng13.uniform(30, 80)
                    x0 = rng13.uniform(0, W-size); y0 = rng13.uniform(0, H-size)
                    cand = [x0, y0, x0+size, y0+size]
                    if not overlaps_any(cand, boxes):
                        bad_box = cand; break
            elif absence_type == "shifted":
                bw = real_box[2]-real_box[0]; bh = real_box[3]-real_box[1]
                dx = rng13.choice([-1, 1]) * bw * rng13.uniform(1.2, 2.0)
                dy = rng13.choice([-1, 1]) * bh * rng13.uniform(1.2, 2.0)
                cand = [real_box[0]+dx, real_box[1]+dy, real_box[2]+dx, real_box[3]+dy]
                ok = 0 <= cand[0] and cand[2] <= W and 0 <= cand[1] and cand[3] <= H
                if ok and not overlaps_any(cand, boxes):
                    bad_box = cand
            else:
                cx = (real_box[0]+real_box[2])/2; cy = (real_box[1]+real_box[3])/2
                scale = rng13.uniform(3.0, 5.0)
                bw = (real_box[2]-real_box[0]) * scale
                bh = (real_box[3]-real_box[1]) * scale
                cand = [cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2]
                if cand[0] >= 0 and cand[2] <= W and cand[1] >= 0 and cand[3] <= H:
                    bad_box = cand
            if bad_box is None:
                continue
            cbox = [max(0, bad_box[0]-m), max(0, bad_box[1]-m),
                   min(W, bad_box[2]+m), min(H, bad_box[3]+m)]
            cb = [int(bad_box[0]-cbox[0]), int(bad_box[1]-cbox[1]),
                 int(bad_box[2]-cbox[0]), int(bad_box[3]-cbox[1])]
            crop = img.crop([int(v) for v in cbox])
            p = (f"A symbol detector claims there is a P&ID equipment "
                f"symbol at [{cb}] in this crop (pixel coords, "
                f"top-left origin). Verify against the pixels: is "
                f"there really a discrete equipment symbol there "
                f"(valve, instrument, fitting)? Answer keep or remove.")
            examples.append({"crop": crop, "prompt": p, "is_real": False})
    return examples

def parse_graphml(path):
    root = ET.parse(path).getroot()
    ns = {"g": "http://graphml.graphdrawing.org/xmlns"}
    keymap = {k.get("id"): k.get("attr.name") for k in root.findall("g:key", ns)}
    nodes, edges = {}, set()
    for node in root.iter("{http://graphml.graphdrawing.org/xmlns}node"):
        vals = {keymap.get(d.get("key"), ""): d.text for d in node.findall("g:data", ns)}
        try:
            nodes[node.get("id")] = [float(vals["xmin"]), float(vals["ymin"]),
                                     float(vals["xmax"]), float(vals["ymax"])]
        except (KeyError, TypeError, ValueError):
            continue
    for e in root.iter("{http://graphml.graphdrawing.org/xmlns}edge"):
        if e.get("source") in nodes and e.get("target") in nodes:
            edges.add(frozenset((e.get("source"), e.get("target"))))
    return nodes, edges

rng12 = random.Random(8181)   # distinct from training (777) and the earlier live eval (4242)
MAX_SPAN, MARGIN = 1400, 80

def build_relation_pool(n_target):
    pool = []
    trees = [OPEN100_DIR]
    if DATASETPID_DIR.exists():
        trees.append(DATASETPID_DIR)
    gmls = []
    for tree in trees:
        gmls.extend(sorted(tree.rglob("*.graphml")))
    rng12.shuffle(gmls)
    for gml in gmls:
        if len(pool) >= n_target:
            break
        png = gml.with_suffix(".png")
        if not png.exists():
            continue
        try:
            nodes, edges = parse_graphml(gml)
        except ET.ParseError:
            continue
        if len(nodes) < 4 or not edges:
            continue
        node_ids = list(nodes)
        want = len(pool) % 2 == 0
        pair = None
        if want:
            pos = [tuple(e) for e in edges]
            rng12.shuffle(pos)
            pair = pos[0] if pos else None
        else:
            for _ in range(50):
                a, b = rng12.sample(node_ids, 2)
                if frozenset((a, b)) not in edges:
                    pair = (a, b); break
        if pair is None:
            continue
        a, b = pair
        ba, bb = nodes[a], nodes[b]
        ux0 = min(ba[0], bb[0]); uy0 = min(ba[1], bb[1])
        ux1 = max(ba[2], bb[2]); uy1 = max(ba[3], bb[3])
        if ux1-ux0 > MAX_SPAN or uy1-uy0 > MAX_SPAN:
            continue
        cx0 = max(0, int(ux0-MARGIN)); cy0 = max(0, int(uy0-MARGIN))
        try:
            img = Image.open(png).convert("RGB")
        except Exception:
            continue
        crop = img.crop((cx0, cy0, int(ux1+MARGIN), int(uy1+MARGIN)))
        a_local = [int(ba[0]-cx0), int(ba[1]-cy0), int(ba[2]-cx0), int(ba[3]-cy0)]
        b_local = [int(bb[0]-cx0), int(bb[1]-cy0), int(bb[2]-cx0), int(bb[3]-cy0)]
        prompt = (f"In this P&ID crop there is a symbol at [{a_local}] "
                 f"and another at [{b_local}] (pixel coordinates, "
                 f"top-left origin). Are these two symbols directly "
                 f"connected to each other? Answer yes or no.")
        pool.append({"crop": crop, "prompt": prompt, "connected": want})
    return pool

stage13_pool = build_stage13_pool(N_PER_TASK)
relation_pool = build_relation_pool(N_PER_TASK)
print(f"stage13_pool: {len(stage13_pool)}  (Gupta frozen TEST sheets)")
print(f"relation_pool: {len(relation_pool)}  (seed-disjoint from training)")

stage13_pool: 120  (Gupta frozen TEST sheets)
relation_pool: 120  (seed-disjoint from training)


## 5. Model callers

In [9]:
import base64, io

from openai import OpenAI
_oa = OpenAI(api_key=OPENAI_API_KEY)

def gpt_generate(image, prompt, max_tokens=2000):
    buf = io.BytesIO(); image.save(buf, format="PNG")
    b64 = base64.standard_b64encode(buf.getvalue()).decode()
    try:
        resp = _oa.chat.completions.create(
            model="gpt-5.5", reasoning_effort=GPT_REASONING_EFFORT,
            max_completion_tokens=max_tokens,
            messages=[{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": prompt},
            ]}])
        return (resp.choices[0].message.content or "").strip()
    except Exception as e:
        print(f"  [api-fail] {type(e).__name__}")
        return ""

from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel
from huggingface_hub import snapshot_download
import random as _random_retry

# ModelScope was tried as a Xet-bypass but REMOVED: (1) its CDN is China-peered,
# measured ~1Mbps from this Colab VM vs ~200Mbps on HF - a 17GB model would take
# ~37h, so it can never be a practical fallback here; (2) a thread-based timeout
# can abandon-and-move-on but can't actually KILL a Python thread, so the
# abandoned download kept running in the background, bled its log lines into
# whatever cell was executing next, and wasted bandwidth/CPU for nothing.
# HF alone, hardened: no force_download (resumes via HF's own .incomplete file
# instead of restarting from 0), more attempts, exponential backoff with jitter.
# Evidence from today: every real occurrence of the Xet 403 cleared within <=7
# attempts - this budget is well above that.
def load_with_retry(loader_fn, max_attempts=20, base_backoff_s=10, max_backoff_s=90):
    last_err = None
    for attempt in range(max_attempts):
        try:
            return loader_fn()
        except Exception as e:
            last_err = e
            backoff = min(max_backoff_s, base_backoff_s * (1.5 ** attempt))
            wait = backoff + _random_retry.uniform(0, backoff * 0.3)
            print(f"  [retry {attempt+1}/{max_attempts}] {type(e).__name__}: "
                 f"{str(e)[:160]} - waiting {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError(f"failed after {max_attempts} retries - HF Xet-bridge "
                       "signing issue persisted longer than usual; rerun this cell") from last_err

processor = load_with_retry(lambda: AutoProcessor.from_pretrained(QWEN_MODEL_ID))
base_model = load_with_retry(lambda: AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda")).eval()

print("Qwen base loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

def _pull(adapter_path):
    local = Path(f"/content/bench_{adapter_path.replace('/', '_')}")
    load_with_retry(lambda: snapshot_download(
        repo_id=CKPT_REPO, repo_type="model", token=HF_TOKEN,
        allow_patterns=[f"{adapter_path}/*"], local_dir=str(local)))
    d = local / adapter_path
    assert (d / "adapter_model.safetensors").exists(), f"missing: {d}"
    return str(d)

stage13_dir = _pull(STAGE13_ADAPTER_PATH)
relation_dir = _pull(RELATION_ADAPTER_PATH)

model = PeftModel.from_pretrained(base_model, stage13_dir, adapter_name="stage13")
model.load_adapter(relation_dir, adapter_name="relation")
print("both adapters attached: 'stage13' and 'relation' "
     "(switch with model.set_adapter(...))")

def qwen_generate(image, prompt, max_tokens=32):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    t = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(t, skip_special_tokens=True).strip()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Qwen base loaded. VRAM: 17.5 GB


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


both adapters attached: 'stage13' and 'relation' (switch with model.set_adapter(...))


In [11]:
processor = load_with_retry(lambda: AutoProcessor.from_pretrained(QWEN_MODEL_ID))
base_model = load_with_retry(lambda: AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda")).eval()
print("Qwen base loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Qwen base loaded. VRAM: 35.4 GB


## 6. Scoring functions

In [12]:
def score_stage13(generate_fn):
    correct = undecided = 0
    for ex in stage13_pool:
        ans = generate_fn(ex["crop"], ex["prompt"]).lower()
        keep = ans.startswith("keep"); remove = ans.startswith("remove")
        if not keep and not remove:
            undecided += 1; continue
        correct += (keep == ex["is_real"])
    n_dec = len(stage13_pool) - undecided
    acc = round(100 * correct / n_dec, 1) if n_dec else None
    return {"acc_%": acc, "undecided": undecided, "n": len(stage13_pool)}

def score_relation(generate_fn):
    correct = undecided = 0
    for ex in relation_pool:
        ans = generate_fn(ex["crop"], ex["prompt"]).lower()
        yes = ans.startswith("yes"); no = ans.startswith("no")
        if not yes and not no:
            undecided += 1; continue
        correct += (yes == ex["connected"])
    n_dec = len(relation_pool) - undecided
    acc = round(100 * correct / n_dec, 1) if n_dec else None
    return {"acc_%": acc, "undecided": undecided, "n": len(relation_pool)}

## 7. Run all six scores

In [13]:
import time
results = {}

t0 = time.time()
results[("gpt-5.5-low", "stage13")] = score_stage13(gpt_generate)
print(f"gpt-5.5-low / stage13 done ({time.time()-t0:.0f}s)")
results[("gpt-5.5-low", "relation")] = score_relation(gpt_generate)
print(f"gpt-5.5-low / relation done ({time.time()-t0:.0f}s)")

model.eval()
with model.disable_adapter():
    results[("qwen-base", "stage13")] = score_stage13(qwen_generate)
    print(f"qwen-base / stage13 done ({time.time()-t0:.0f}s)")
    results[("qwen-base", "relation")] = score_relation(qwen_generate)
    print(f"qwen-base / relation done ({time.time()-t0:.0f}s)")

model.set_adapter("stage13")
results[("qwen-v3-stage13", "stage13")] = score_stage13(qwen_generate)
print(f"qwen-v3-stage13 / stage13 done ({time.time()-t0:.0f}s)")

model.set_adapter("relation")
results[("qwen-v3-relation", "relation")] = score_relation(qwen_generate)
print(f"qwen-v3-relation / relation done ({time.time()-t0:.0f}s)")

gpt-5.5-low / stage13 done (575s)
gpt-5.5-low / relation done (1369s)
qwen-base / stage13 done (1398s)


2026-07-14 10:20:24,559 | WARNING | modelscope_hub.download | Hash validation failed for model-00004-of-00004.safetensors, retrying (1/3)


model-00004-of-00004.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

qwen-base / relation done (1420s)
qwen-v3-stage13 / stage13 done (1452s)
qwen-v3-relation / relation done (1598s)


## 8. Verdict table + push to HF

In [14]:
print(f"{'model':20s}{'stage13 (n=' + str(N_PER_TASK) + ')':>22s}"
     f"{'relation (n=' + str(N_PER_TASK) + ')':>24s}")
print("-" * 66)
for m in ["gpt-5.5-low", "qwen-base", "qwen-v3-stage13", "qwen-v3-relation"]:
    r13 = results.get((m, "stage13"))
    r12 = results.get((m, "relation"))
    def fmt(r):
        if r is None:
            return "n/a"
        a = r["acc_%"]
        return (str(a) + "%" if a is not None else "n/a") + f" (u{r['undecided']})"
    print(f"{m:20s}{fmt(r13):>22s}{fmt(r12):>24s}")
print()
print("baselines: stage13 chance=50%  relation chance=50%")
print("prior references: v2 adapter stage13=35% (below chance); "
     "v2 adapter relation=84% (n=25, noisy)")

import csv, datetime
from huggingface_hub import HfApi
RESULTS_LOCAL = Path("/content/per_stage_v3_results.csv")
FIELDS = ["timestamp", "model", "task", "acc_%", "undecided", "n"]
try:
    prev = hf_hub_download(repo_id=DATA_REPO, filename=RESULTS_PATH_IN_REPO,
                           repo_type="dataset", token=HF_TOKEN)
    RESULTS_LOCAL.write_text(Path(prev).read_text())
except Exception:
    with open(RESULTS_LOCAL, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=FIELDS).writeheader()
with open(RESULTS_LOCAL, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    ts = datetime.datetime.utcnow().isoformat(timespec="seconds")
    for (model_name, task), r in results.items():
        w.writerow({"timestamp": ts, "model": model_name, "task": task,
                   "acc_%": r["acc_%"], "undecided": r["undecided"], "n": r["n"]})
HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=str(RESULTS_LOCAL),
    path_in_repo=RESULTS_PATH_IN_REPO, repo_id=DATA_REPO, repo_type="dataset")
print("\nresults pushed to HF.")

model                      stage13 (n=120)        relation (n=120)
------------------------------------------------------------------
gpt-5.5-low                     66.7% (u0)              72.5% (u0)
qwen-base                       45.0% (u0)              80.0% (u0)
qwen-v3-stage13                 89.2% (u0)                     n/a
qwen-v3-relation                       n/a              89.2% (u0)

baselines: stage13 chance=50%  relation chance=50%
prior references: v2 adapter stage13=35% (below chance); v2 adapter relation=84% (n=25, noisy)


/tmp/ipykernel_1156/3575746670.py:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.datetime.utcnow().isoformat(timespec="seconds")



results pushed to HF.


2026-07-14 10:36:22,508 | WARNING | modelscope_hub.download | Hash validation failed for model-00003-of-00004.safetensors, retrying (1/3)


model-00003-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

In [2]:
import random

## 9. Text extraction (OCR tie-break) — n=120, added post-hoc

Same task definition as yesterday's `5_ocr_tiebreak` stage (from
`ThreeStages_GPT55_vs_QwenV2_vs_QwenBase.ipynb`), reused verbatim so this is an
apples-to-apples re-run, just at a trustworthy sample size: Tesseract OCR pulls a
real text label off a Gupta TEST sheet, one single-character-corrupted variant is
generated, the two are randomly assigned to A/B, and the model picks which one
matches the pixels exactly (chance = 50%). Yesterday's "100%" ran at n=25 and was
never saved anywhere in the repo — this n=120 run is the first trustworthy number
for this task. No v3 adapter exists for reading, so only gpt-5.5-low and qwen-base
are scored (reuses the already-loaded model/processor from Section 5 — no reload,
no re-download).

In [11]:
import pytesseract, random, csv, datetime
from huggingface_hub import HfApi

rng_read = random.Random(2222)

def corrupt_ocr(word):
    repl_map = {"0":"8","1":"7","5":"6","B":"8","O":"0","I":"1"}
    i = rng_read.randrange(len(word))
    repl = repl_map.get(word[i], rng_read.choice("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ-"))
    while repl == word[i]:
        repl = rng_read.choice("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ-")
    return word[:i] + repl + word[i+1:]

def build_reading_pool(n_target):
    pool = []
    test_sheets = sorted((GUPTA_RAW / "sheets" / "test").glob("*.*"))
    rng_read.shuffle(test_sheets)
    for sheet in test_sheets:
        if len(pool) >= n_target:
            break
        img = Image.open(sheet).convert("RGB")
        tile = img.crop((0, 0, min(2048, img.width), min(2048, img.height)))
        data = pytesseract.image_to_data(tile, output_type=pytesseract.Output.DICT)
        for i in range(len(data["text"])):
            if len(pool) >= n_target:
                break
            t = data["text"][i].strip()
            if (len(t) >= 4 and int(data["conf"][i]) >= 70
                    and (any(c.isdigit() for c in t) or "-" in t)):
                m = 20
                crop = tile.crop((max(0, data["left"][i]-m), max(0, data["top"][i]-m),
                                  data["left"][i]+data["width"][i]+m, data["top"][i]+data["height"][i]+m))
                a_is_true = rng_read.random() < 0.5
                A, B = (t, corrupt_ocr(t)) if a_is_true else (corrupt_ocr(t), t)
                pool.append({"crop": crop, "a_is_true": a_is_true,
                    "prompt": (f"This crop shows one text label from an engineering drawing. Two OCR "
                              f"readings were proposed:\nA: {A}\nB: {B}\nWhich exactly matches the "
                              f"pixels? Answer A or B only.")})
    return pool

reading_pool = build_reading_pool(N_PER_TASK)
print(f"reading_pool: {len(reading_pool)}  (Gupta frozen TEST sheets, OCR tie-break)")

def score_reading(generate_fn):
    correct = undecided = 0
    for ex in reading_pool:
        ans = generate_fn(ex["crop"], ex["prompt"]).strip().upper()
        if not ans or ans[0] not in "AB":
            undecided += 1; continue
        correct += (ans[0] == "A") == ex["a_is_true"]
    n_dec = len(reading_pool) - undecided
    acc = round(100 * correct / n_dec, 1) if n_dec else None
    return {"acc_%": acc, "undecided": undecided, "n": len(reading_pool)}

reading_results = {}
reading_results["gpt-5.5-low"] = score_reading(gpt_generate)
print("gpt-5.5-low / reading done")
with model.disable_adapter():
    reading_results["qwen-base"] = score_reading(qwen_generate)
print("qwen-base / reading done")

print()
print(f"{'model':20s}{'reading (n=' + str(N_PER_TASK) + ')':>22s}")
print("-" * 42)
for m, r in reading_results.items():
    val = f"{r['acc_%']}%" if r["acc_%"] is not None else "n/a"
    print(f"{m:20s}{val:>16s} (u{r['undecided']})")
print()
print("baseline: chance=50% (A/B tie-break)")
print("prior reference: yesterday reported ~100% at n=25 for qwen - never saved, "
     "and n=25 is the exact sample size already shown unreliable on this project "
     "(the relation task's noisy 84%-vs-80% swing) - this n=120 result supersedes it")

# append to the same results CSV as the main run
try:
    prev = hf_hub_download(repo_id=DATA_REPO, filename=RESULTS_PATH_IN_REPO,
                           repo_type="dataset", token=HF_TOKEN)
    Path("/content/per_stage_v3_results.csv").write_text(Path(prev).read_text())
except Exception:
    pass
with open("/content/per_stage_v3_results.csv", "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["timestamp", "model", "task", "acc_%", "undecided", "n"])
    ts = datetime.datetime.utcnow().isoformat(timespec="seconds")
    for model_name, r in reading_results.items():
        w.writerow({"timestamp": ts, "model": model_name, "task": "reading",
                   "acc_%": r["acc_%"], "undecided": r["undecided"], "n": r["n"]})
HfApi(token=HF_TOKEN).upload_file(path_or_fileobj="/content/per_stage_v3_results.csv",
    path_in_repo=RESULTS_PATH_IN_REPO, repo_id=DATA_REPO, repo_type="dataset")
print("\nreading results appended to HF.")

reading_pool: 65  (Gupta frozen TEST sheets, OCR tie-break)
gpt-5.5-low / reading done
qwen-base / reading done

model                      reading (n=120)
------------------------------------------
gpt-5.5-low                    98.5% (u0)
qwen-base                     100.0% (u0)

baseline: chance=50% (A/B tie-break)
prior reference: yesterday reported ~100% at n=25 for qwen - never saved, and n=25 is the exact sample size already shown unreliable on this project (the relation task's noisy 84%-vs-80% swing) - this n=120 result supersedes it


/tmp/ipykernel_26471/3431035830.py:83: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.datetime.utcnow().isoformat(timespec="seconds")



reading results appended to HF.


## 10. Qualitative sample gallery — real crops + actual model outputs

A handful of real examples per task, each showing: the actual crop image, the
prompt, the ground-truth answer, and what every relevant model actually said
(not just correct/incorrect — the raw text). Reuses everything already in
memory (pools, model, processor, gpt_generate, qwen_generate) — no re-run, no
re-download. Saves a self-contained HTML file and triggers a browser download.

In [13]:
import base64, io as _io

N_SAMPLE = 4

def _b64(img):
    buf = _io.BytesIO(); img.save(buf, format="PNG")
    return base64.standard_b64encode(buf.getvalue()).decode()

def _sample_task(pool, expected_fn, adapter_name):
    if not pool:
        return []
    samples = pool[:N_SAMPLE]
    rows = []
    for ex in samples:
        rows.append({"crop_b64": _b64(ex["crop"]), "prompt": ex["prompt"],
                     "expected": expected_fn(ex), "answers": {}})
    for row, ex in zip(rows, samples):
        row["answers"]["gpt-5.5-low"] = gpt_generate(ex["crop"], ex["prompt"])
    with model.disable_adapter():
        for row, ex in zip(rows, samples):
            row["answers"]["qwen-base"] = qwen_generate(ex["crop"], ex["prompt"])
    if adapter_name:
        model.set_adapter(adapter_name)
        for row, ex in zip(rows, samples):
            row["answers"][f"qwen-v3-{adapter_name}"] = qwen_generate(ex["crop"], ex["prompt"])
    return rows

gallery_stage13 = _sample_task(stage13_pool, lambda ex: "keep" if ex["is_real"] else "remove", "stage13")
gallery_relation = _sample_task(relation_pool, lambda ex: "yes" if ex["connected"] else "no", "relation")
gallery_reading = _sample_task(reading_pool if "reading_pool" in globals() else [],
                               lambda ex: "A" if ex["a_is_true"] else "B", None)

print(f"gallery built: {len(gallery_stage13)} stage13, {len(gallery_relation)} relation, "
     f"{len(gallery_reading)} reading samples")

def _render_gallery_html(sections):
    parts = ['''<!DOCTYPE html><html><head><meta charset="utf-8">
<title>Qualitative sample gallery</title>
<style>
body{font-family:system-ui,-apple-system,"Segoe UI",sans-serif;background:#f9f9f7;
color:#0b0b0b;padding:24px;max-width:900px;margin:0 auto}
h2{margin-top:36px}
.card{background:#fcfcfb;border:1px solid rgba(11,11,11,0.1);border-radius:10px;
padding:16px;margin-bottom:16px;display:flex;gap:16px}
.card img{max-width:260px;max-height:220px;border-radius:6px;
border:1px solid rgba(11,11,11,0.1);object-fit:contain}
.meta{flex:1;font-size:13px;line-height:1.6;min-width:0}
.prompt{color:#52514e;white-space:pre-wrap;font-size:12px;margin-bottom:8px}
.expected{font-weight:600}
.answer-row{display:flex;justify-content:space-between;gap:12px;
border-top:1px solid #e1e0d9;padding:6px 0}
.answer-row .label{color:#52514e;flex:none}
.answer-row .val{text-align:right;word-break:break-word}
.correct{color:#0ca30c;font-weight:600}
.wrong{color:#d03b3b;font-weight:600}
</style></head><body>''']
    for title, rows in sections:
        if not rows:
            continue
        parts.append(f"<h2>{title}</h2>")
        for row in rows:
            parts.append('<div class="card">')
            parts.append(f'<img src="data:image/png;base64,{row["crop_b64"]}">')
            parts.append('<div class="meta">')
            parts.append(f'<div class="prompt">{row["prompt"]}</div>')
            parts.append(f'<div class="expected">expected: {row["expected"]}</div>')
            for label, ans in row["answers"].items():
                is_correct = ans.strip().lower().startswith(row["expected"].lower())
                cls = "correct" if is_correct else "wrong"
                parts.append(f'<div class="answer-row"><span class="label">{label}</span>'
                             f'<span class="val {cls}">{ans!r}</span></div>')
            parts.append("</div></div>")
    parts.append("</body></html>")
    return "\n".join(parts)

gallery_html = _render_gallery_html([
    ("Stage 13 — entity validation", gallery_stage13),
    ("Relation — connectivity", gallery_relation),
    ("Reading — OCR tie-break", gallery_reading),
])

with open("/content/qualitative_gallery.html", "w") as f:
    f.write(gallery_html)
print("saved: /content/qualitative_gallery.html")

try:
    from google.colab import files as _colab_files
    _colab_files.download("/content/qualitative_gallery.html")
except Exception as e:
    print(f"auto-download unavailable ({type(e).__name__}) - "
         "open the Colab file browser and download /content/qualitative_gallery.html manually")

gallery built: 4 stage13, 4 relation, 4 reading samples
saved: /content/qualitative_gallery.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. General (v2, mixed-task) adapter — same pools, true apples-to-apples

Loads the OLD v2 adapter (`v2/latest` — trained across mixed tasks: count,
relation, reading, stage13 together, before the per-stage v3 redesign) as a
THIRD named adapter on the already-loaded model, and scores it on the exact
same `stage13_pool` / `relation_pool` / `reading_pool` already built — same
n=120/65, same examples, same scoring functions as everything above. This
replaces the old informal "v2 stage13=35%, v2 relation=84% (n=25, noisy)"
reference points with real numbers on this run's held-out data. No new
download except the v2 checkpoint itself (~210MB).

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download as _snap_v2

_v2_local = Path("/content/bench_v2")
load_with_retry(lambda: _snap_v2(
    repo_id=CKPT_REPO, repo_type="model", token=HF_TOKEN,
    allow_patterns=["v2/latest/*"], local_dir=str(_v2_local)))
_v2_dir = _v2_local / "v2" / "latest"
assert (_v2_dir / "adapter_model.safetensors").exists(), f"missing: {_v2_dir}"

model.load_adapter(str(_v2_dir), adapter_name="v2")
model.set_adapter("v2")
print("loaded v2 (general, mixed-task) adapter as a third named adapter -> active")

v2_results = {}
v2_results["stage13"] = score_stage13(qwen_generate)
print(f"qwen-v2 / stage13 done")
v2_results["relation"] = score_relation(qwen_generate)
print(f"qwen-v2 / relation done")
if "reading_pool" in globals() and reading_pool:
    v2_results["reading"] = score_reading(qwen_generate)
    print(f"qwen-v2 / reading done")

print()
print(f"{'task':20s}{'qwen-v2 (general)':>20s}")
print("-" * 40)
for task, r in v2_results.items():
    val = f"{r['acc_%']}%" if r["acc_%"] is not None else "n/a"
    print(f"{task:20s}{val:>14s} (u{r['undecided']}, n={r['n']})")
print()
print("supersedes the old informal references (v2 stage13~35%, v2 relation~84%, "
     "both n=25/unclear pool) with real numbers on this run's actual held-out data")

# append to the same results CSV
import csv, datetime
from huggingface_hub import HfApi, hf_hub_download as _hfd_v2
try:
    prev = _hfd_v2(repo_id=DATA_REPO, filename=RESULTS_PATH_IN_REPO,
                   repo_type="dataset", token=HF_TOKEN)
    Path("/content/per_stage_v3_results.csv").write_text(Path(prev).read_text())
except Exception:
    pass
with open("/content/per_stage_v3_results.csv", "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["timestamp", "model", "task", "acc_%", "undecided", "n"])
    ts = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    for task, r in v2_results.items():
        w.writerow({"timestamp": ts, "model": "qwen-v2-general", "task": task,
                   "acc_%": r["acc_%"], "undecided": r["undecided"], "n": r["n"]})
HfApi(token=HF_TOKEN).upload_file(path_or_fileobj="/content/per_stage_v3_results.csv",
    path_in_repo=RESULTS_PATH_IN_REPO, repo_id=DATA_REPO, repo_type="dataset")
print("\nv2 results appended to HF.")

# restore a v3 adapter as active before any further use, so v2 doesn't linger active
model.set_adapter("stage13")
print("switched back to 'stage13' (v3) adapter.")